# Indoor Air Quality Anomaly Detection using LSTM Autoencoder
**EEE G513 — Machine Learning for Electronics Engineers**  
Nikhil Kumar Kambampati (2025H1400104P) · Akash Pasala (2025H1400043P)  
Instructor: Dr. Ritesh Vyas · BITS Pilani, Pilani Campus

This is an **independently reconstructed, end-to-end Colab workflow** for the project, **not an export of the original working notebook**. It downloads the six required DALTON CSV files, preprocesses the data, trains the LSTM and Isolation Forest, and evaluates both on the **same controlled synthetic samples**.

**Scientific caveat:** DALTON activity annotations are not verified anomaly labels. Synthetic accuracy/F1 are **not real-world anomaly detection accuracy**; unmodified windows are only *assumed* normal in the controlled evaluation.

> Before running: Runtime → Change runtime type → T4 GPU. Training regenerates a new checkpoint; numerical values may differ from the prior run.

In [ ]:
# 1. Imports and configuration
from pathlib import Path
import os, subprocess, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import joblib
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
PROJECT_DIR=Path('/content/IAQ_Anomaly_Detection')
REPO_DIR=PROJECT_DIR/'DALTON'
H1_DIR=REPO_DIR/'Processed'/'H1'
MODEL_DIR=PROJECT_DIR/'models'
RESULTS_DIR=PROJECT_DIR/'results'
PLOTS_DIR=PROJECT_DIR/'plots'
for folder in [PROJECT_DIR,MODEL_DIR,RESULTS_DIR,PLOTS_DIR]:
    folder.mkdir(parents=True,exist_ok=True)
FEATURES=['T','H','PMS1','PMS2_5','PMS10','CO2','NO2','CO','VoC','C2H5OH']
WINDOW_SIZE=60
TRAIN_DATES=['2023_05_28','2023_05_29','2023_05_30']
VAL_DATES=['2023_05_31']
TEST_DATES=['2023_06_04','2023_06_05']
ALL_DATES=TRAIN_DATES+VAL_DATES+TEST_DATES
print('TensorFlow:',tf.__version__,'GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. Download only the six DALTON kitchen CSVs + activity annotations through Git LFS
# (Avoid pulling all 1,335 large processed files.)
selected_paths=[f'Processed/H1/{day}/41_Kitchen.csv' for day in ALL_DATES]
selected_paths.append('Metadata/Annotations_cleaned.csv')

def real_csv(path):
    if not path.exists() or path.stat().st_size<1000: return False
    with path.open('rb') as f: return f.readline().startswith(b'ts,')

if not all(real_csv(REPO_DIR/x) for x in selected_paths):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','git-lfs'],check=True)
    subprocess.run(['git','lfs','install'],check=True)
    if not (REPO_DIR/'.git').exists():
        if REPO_DIR.exists():
            raise RuntimeError('Repository directory exists but is not a git clone; inspect before deleting.')
        environment=os.environ.copy(); environment['GIT_LFS_SKIP_SMUDGE']='1'
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/prasenjit52282/dalton-dataset.git',
                        str(REPO_DIR)],env=environment,check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'lfs','pull',
                    '--include='+','.join(selected_paths)],check=True)
assert all(real_csv(REPO_DIR/x) for x in selected_paths), 'Some Git LFS CSV data were not downloaded.'
print('Required DALTON CSV files available:',len(selected_paths))

In [ ]:
# 3. Load and filter recordings, preserving continuous valid 1-Hz segments
# These quality criteria are an explicit design choice, not independent validity certification.
def load_clean_recordings(dates):
    output=[]
    for day in dates:
        path=H1_DIR/day/'41_Kitchen.csv'
        df=pd.read_csv(path,low_memory=False)
        df['ts']=pd.to_datetime(df['ts'],errors='coerce')
        for f in FEATURES: df[f]=pd.to_numeric(df[f],errors='coerce')
        usable=(pd.to_numeric(df['Valid'],errors='coerce').eq(1)
             & pd.to_numeric(df['Valid_CO2'],errors='coerce').eq(1)
             & np.isfinite(df[FEATURES]).all(axis=1)
             & ~df[FEATURES].eq(0).all(axis=1)
             & df['ts'].notna())
        clean=df.loc[usable,['ts']+FEATURES].copy()
        clean['recording_date']=day
        gaps=(clean.index.to_series().diff().ne(1)
              | clean['ts'].diff().ne(pd.Timedelta(seconds=1)))
        clean['segment_id']=gaps.cumsum()
        output.append(clean)
        print(day, len(clean), 'rows,',clean.segment_id.nunique(),'segments')
    return output

train_recordings=load_clean_recordings(TRAIN_DATES)
val_recordings=load_clean_recordings(VAL_DATES)
test_recordings=load_clean_recordings(TEST_DATES)

In [ ]:
# 4. Fit MinMaxScaler on training readings only; window within uninterrupted segments
train_values=pd.concat([df[FEATURES] for df in train_recordings],ignore_index=True)
scaler=MinMaxScaler().fit(train_values)
joblib.dump(scaler,MODEL_DIR/'minmax_scaler.joblib')

def create_sequences(recordings,scaler,window_size=60):
    arrays=[]; records=[]
    for recording in recordings:
        day=recording['recording_date'].iloc[0]
        for segment_id,segment in recording.groupby('segment_id',sort=False):
            if len(segment)<window_size: continue
            values=scaler.transform(segment[FEATURES]).astype(np.float32)
            windows=np.lib.stride_tricks.sliding_window_view(
                values,window_shape=window_size,axis=0).transpose(0,2,1).copy()
            arrays.append(windows)
            timestamps=segment.ts.to_numpy()[window_size-1:]
            records.append(pd.DataFrame({'date':day,'timestamp':timestamps,'segment_id':int(segment_id)}))
    return np.concatenate(arrays,axis=0),pd.concat(records,ignore_index=True)

X_train,train_metadata=create_sequences(train_recordings,scaler,WINDOW_SIZE)
X_val,val_metadata=create_sequences(val_recordings,scaler,WINDOW_SIZE)
X_test,test_metadata=create_sequences(test_recordings,scaler,WINDOW_SIZE)
print('Shapes:',X_train.shape,X_val.shape,X_test.shape)
assert X_train.shape==(246009,60,10)
assert X_val.shape==(86282,60,10)
assert X_test.shape==(172564,60,10)

In [ ]:
# 5. Define and train the independent 15,242-parameter LSTM Autoencoder
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

inputs=Input(shape=(WINDOW_SIZE,len(FEATURES)),name='sensor_input')
x=LSTM(32,return_sequences=True,name='encoder_lstm_1')(inputs)
x=LSTM(16,name='encoder_lstm_2')(x)
x=RepeatVector(WINDOW_SIZE,name='repeat_latent')(x)
x=LSTM(32,return_sequences=True,name='decoder_lstm')(x)
outputs=TimeDistributed(Dense(len(FEATURES)),name='reconstructed_features')(x)
autoencoder=Model(inputs,outputs,name='IAQ_LSTM_Autoencoder')
assert autoencoder.count_params()==15242

model_path=MODEL_DIR/'best_iaq_lstm_autoencoder.keras'
if model_path.exists():
    print('Loading existing independent-model checkpoint:',model_path)
    best_model=tf.keras.models.load_model(str(model_path))
else:
    autoencoder.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mse')
    history=autoencoder.fit(
        X_train,X_train,validation_data=(X_val,X_val),
        epochs=30,batch_size=256,shuffle=True,verbose=1,
        callbacks=[
            EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True,verbose=1),
            ModelCheckpoint(str(model_path),monitor='val_loss',save_best_only=True,verbose=1),
            ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=3,min_lr=1e-5,verbose=1)
        ])
    pd.DataFrame(history.history).to_csv(RESULTS_DIR/'training_history.csv',index=False)
    best_model=tf.keras.models.load_model(str(model_path))
print('Loaded model params:',best_model.count_params())
assert best_model.count_params()==15242, 'Wrong checkpoint: expected independent 15,242-param model.'

In [ ]:
# 6. Reconstruction MSE and 99th-percentile validation threshold
# Batching avoids allocating all predictions in RAM.
def reconstruction_errors(model,X,batch_size=256):
    chunks=[]
    for start in range(0,len(X),batch_size):
        batch=X[start:start+batch_size]
        reconstructed=model(batch,training=False).numpy()
        chunks.append(np.mean(np.square(batch-reconstructed),axis=(1,2)))
    return np.concatenate(chunks)

train_errors=reconstruction_errors(best_model,X_train)
val_errors=reconstruction_errors(best_model,X_val)
test_errors=reconstruction_errors(best_model,X_test)
anomaly_threshold=float(np.percentile(val_errors,99))
test_anomalies=test_errors>anomaly_threshold
print('LSTM threshold:',anomaly_threshold,'test flagged:',int(test_anomalies.sum()),'rate %:',100*test_anomalies.mean())
test_results=test_metadata.copy()
test_results['reconstruction_error']=test_errors
test_results['predicted_anomaly']=test_anomalies.astype(int)
test_results['threshold']=anomaly_threshold
test_results.to_csv(RESULTS_DIR/'lstm_test_predictions.csv',index=False)

In [ ]:
# 7. Group consecutive flagged windows into distinct test events
flagged=test_results.loc[test_results.predicted_anomaly.eq(1)].sort_values(['date','timestamp']).copy()
flagged['timestamp']=pd.to_datetime(flagged['timestamp'])
new_event=(flagged.date.ne(flagged.date.shift())
           | flagged.timestamp.diff().gt(pd.Timedelta(seconds=1)))
flagged['event_id']=new_event.cumsum()
events=flagged.groupby('event_id').agg(
    date=('date','first'),first_flagged=('timestamp','min'),last_flagged=('timestamp','max'),
    flagged_windows=('timestamp','size'),peak_error=('reconstruction_error','max')).reset_index()
events.to_csv(RESULTS_DIR/'lstm_anomaly_events.csv',index=False)
print('Event count:',len(events))
display(events.sort_values('peak_error',ascending=False).head(5))

In [ ]:
# 8. Compare events with activity annotations (association, NOT causal proof or recall)
annotation_file=REPO_DIR/'Metadata'/'Annotations_cleaned.csv'
annotations=pd.read_csv(annotation_file,low_memory=False)
annotations['ts']=pd.to_datetime(annotations['ts'],errors='coerce')
h1_activities=annotations[
    annotations['Site'].astype(str).str.strip().eq('H1')
    & annotations['ts'].dt.strftime('%Y_%m_%d').isin(TEST_DATES)
].copy()
results=test_results.copy(); results['timestamp']=pd.to_datetime(results.timestamp)
matched=[]
for _,activity in h1_activities.iterrows():
    nearby=results.loc[results.timestamp.between(
        activity.ts-pd.Timedelta(minutes=5),activity.ts+pd.Timedelta(minutes=5))]
    matched.append({'ts':activity.ts,'label':activity['Label'],
                    'flagged_nearby':bool(nearby.predicted_anomaly.any())})
activity_matches=pd.DataFrame(matched)
activity_matches.to_csv(RESULTS_DIR/'activity_annotation_association.csv',index=False)
print('Annotations:',len(activity_matches),'with nearby flags:',int(activity_matches.flagged_nearby.sum()))

In [ ]:
# 9. Isolation Forest baseline on 40 window summary features (mean/std/min/max)
def statistical_features(X):
    return np.concatenate([X.mean(axis=1),X.std(axis=1),
                           X.min(axis=1),X.max(axis=1)],axis=1).astype(np.float32)
train_idx=[]
for day in TRAIN_DATES:
    indices=np.flatnonzero(train_metadata.date.to_numpy()==day)
    train_idx.extend(indices[::WINDOW_SIZE])
train_idx=np.array(train_idx,dtype=int)
X_train_if_raw=statistical_features(X_train[train_idx])
X_val_if_raw=statistical_features(X_val)
X_test_if_raw=statistical_features(X_test)
if_scaler=StandardScaler().fit(X_train_if_raw)
X_train_if=if_scaler.transform(X_train_if_raw)
X_val_if=if_scaler.transform(X_val_if_raw)
X_test_if=if_scaler.transform(X_test_if_raw)
if_model=IsolationForest(n_estimators=200,max_samples=min(2048,len(X_train_if)),
                         contamination='auto',random_state=42,n_jobs=-1)
if_model.fit(X_train_if)
if_threshold=float(np.percentile(-if_model.score_samples(X_val_if),99))
if_test_scores=-if_model.score_samples(X_test_if)
if_test_flags=if_test_scores>if_threshold
print('IF training shape:',X_train_if.shape,'threshold:',if_threshold,
      'test flags:',int(if_test_flags.sum()),'rate %:',100*if_test_flags.mean())
joblib.dump(if_model,MODEL_DIR/'isolation_forest_baseline.joblib')
joblib.dump(if_scaler,MODEL_DIR/'isolation_forest_scaler.joblib')

## Controlled paired synthetic evaluation

We use **2,000 randomly selected, non-overlapping original test windows** (assumed normal, though some may actually be unusual) and 2,000 **modified copies** containing large artificial spikes, shifts, or noise. This setup measures controlled sensitivity and should **not** be described as verified real-world anomaly accuracy. Both models evaluate the *same* examples.

In [ ]:
# 10. Recreate the paired synthetic evaluation used in the original project
rng_selection=np.random.default_rng(2026)
selected_candidates=[]
for day in sorted(test_metadata.date.unique()):
    indices=np.flatnonzero(test_metadata.date.to_numpy()==day)
    selected_candidates.extend(indices[::WINDOW_SIZE])
selected_indices=np.sort(rng_selection.choice(np.array(selected_candidates,dtype=int),size=2000,replace=False))
X_original=X_test[selected_indices].copy()
X_modified=X_original.copy()
rng=np.random.default_rng(2026)
types=['spike','sustained_shift','noise']; synthetic_types=[]
for i in range(len(X_modified)):
    typ=types[i%3]; synthetic_types.append(typ)
    features=rng.choice(X_modified.shape[2],size=int(rng.integers(1,4)),replace=False)
    if typ=='spike':
        dur=int(rng.integers(5,11)); start=int(rng.integers(0,60-dur+1))
        X_modified[i,start:start+dur,features] += rng.uniform(0.5,1.0)
    elif typ=='sustained_shift':
        dur=int(rng.integers(20,41)); start=int(rng.integers(0,60-dur+1))
        X_modified[i,start:start+dur,features] += rng.uniform(0.25,0.6)
    else:
        X_modified[i][:,features] += rng.normal(0,0.3,size=(60,len(features)))
X_synthetic=np.concatenate([X_original,X_modified],axis=0).astype(np.float32)
y_synthetic=np.array([0]*2000+[1]*2000,dtype=int)
sample_type=np.array(['original']*2000+synthetic_types)
order=rng.permutation(len(y_synthetic))
X_synthetic=X_synthetic[order];y_synthetic=y_synthetic[order];sample_type=sample_type[order]
print('Evaluation set:',X_synthetic.shape,'labels:',np.bincount(y_synthetic))

In [ ]:
# 11. Compare both models on the same synthetic windows and unchanged thresholds
lstm_syn_errors=reconstruction_errors(best_model,X_synthetic)
lstm_pred=(lstm_syn_errors>anomaly_threshold).astype(int)
if_syn_features=if_scaler.transform(statistical_features(X_synthetic))
if_syn_scores=-if_model.score_samples(if_syn_features)
if_pred=(if_syn_scores>if_threshold).astype(int)

def summarize_model(name,pred):
    cm=confusion_matrix(y_synthetic,pred,labels=[0,1]);tn,fp,fn,tp=cm.ravel()
    return {'Model':name,'Accuracy (%)':100*accuracy_score(y_synthetic,pred),
            'Precision (%)':100*precision_score(y_synthetic,pred,zero_division=0),
            'Recall (%)':100*recall_score(y_synthetic,pred,zero_division=0),
            'F1 (%)':100*f1_score(y_synthetic,pred,zero_division=0),
            'TN':int(tn),'FP_assumed':int(fp),'FN':int(fn),'TP':int(tp)}
comparison=pd.DataFrame([summarize_model('Isolation Forest',if_pred),
                         summarize_model('LSTM Autoencoder',lstm_pred)])
display(comparison)
comparison.to_csv(RESULTS_DIR/'paired_synthetic_model_comparison.csv',index=False)
by_type=pd.DataFrame({'type':sample_type,'lstm_flag':lstm_pred,'if_flag':if_pred}).groupby('type').agg(
    count=('lstm_flag','size'),lstm_flag_rate=('lstm_flag','mean'),if_flag_rate=('if_flag','mean'))
display(by_type)
by_type.to_csv(RESULTS_DIR/'paired_synthetic_by_type.csv')

In [ ]:
# 12. Feature-wise MSE contributions in flagged LSTM test sequences
sum_errors=np.zeros(len(FEATURES),dtype=np.float64); flagged_count=0
for start in range(0,len(X_test),256):
    batch=X_test[start:start+256]
    batch_flags=test_anomalies[start:start+256]
    if not np.any(batch_flags):continue
    rec=best_model(batch,training=False).numpy()
    per_feature=np.mean(np.square(batch-rec),axis=1)
    sum_errors+=per_feature[batch_flags].sum(axis=0)
    flagged_count+=int(np.sum(batch_flags))
mean_feature_error=sum_errors/flagged_count
feature_table=pd.DataFrame({'feature':FEATURES,'mean_mse':mean_feature_error,
                            'contribution_pct':100*mean_feature_error/mean_feature_error.sum()})
feature_table=feature_table.sort_values('contribution_pct',ascending=False)
display(feature_table)
feature_table.to_csv(RESULTS_DIR/'feature_contributions.csv',index=False)

In [ ]:
# 13. Save figures and reproducibility configuration
fig,ax=plt.subplots(figsize=(8,4))
comparison.set_index('Model')[['Accuracy (%)','Precision (%)','Recall (%)','F1 (%)']].T.plot.bar(ax=ax)
ax.set_ylim(0,105);ax.set_ylabel('Percentage (%)');ax.set_title('Controlled synthetic evaluation')
ax.tick_params(axis='x',rotation=0);ax.legend(fontsize=8)
plt.tight_layout();fig.savefig(PLOTS_DIR/'model_comparison.png',dpi=240);plt.show()

fig,ax=plt.subplots(figsize=(8,4))
ax.bar(feature_table.feature,feature_table.contribution_pct)
ax.set_ylabel('Share of feature-wise reconstruction MSE (%)')
ax.set_title('Flagged test window feature contributions')
ax.tick_params(axis='x',rotation=45)
plt.tight_layout();fig.savefig(PLOTS_DIR/'feature_contributions.png',dpi=240);plt.show()

configuration={'model':'LSTM AE 32-16-32, 15,242 parameters',
               'dataset':'DALTON H1 41_Kitchen.csv',
               'window':60,'features':FEATURES,'train_dates':TRAIN_DATES,
               'val_dates':VAL_DATES,'test_dates':TEST_DATES,
               'threshold_policy':'99th percentile of validation scores',
               'lstm_threshold':anomaly_threshold,'if_threshold':if_threshold,
               'note':'Synthetic metrics are not verified real-world anomaly classification performance.'}
(PROJECT_DIR/'experiment_config.json').write_text(json.dumps(configuration,indent=2))
print('Saved outputs to:',PROJECT_DIR)

## Deliverables and limitations

- **Artifacts:** `models/*.keras` and `models/*.joblib`; `results/*.csv`; `plots/*.png`; `experiment_config.json`.
- **Dataset:** downloaded from the [DALTON repository](https://github.com/prasenjit52282/dalton-dataset). The repository does **not** redistribute the data.
- **Fairness:** the two algorithms share a chronological date split, validation percentile, and paired synthetic evaluation; Isolation Forest consumes window summary statistics, while LSTM consumes the full sequence.
- **Caution:** no expert-verified real anomaly labels are available. Synthetic recall can be high for large perturbations. Activity coincidence is not causation.
- **Original historical results:** see `results/reported_reference_metrics.json`. Newly trained runs may produce slightly different values.